# GeoEnrichment User Guide

## What is GeoEnrichment

The `arcgis.geoenrichment` module provides access to Business Analyst’s demographic and jurisdictional areas data through enrichment, standard geography queries and reporting. The source for analysis can be either local or remote. Using a local source requires ArcGIS Pro with the Business Analyst extension and at least one country’s data pack to be installed. A remote source requires access to a property configured Web GIS. A Web GIS can either be ArcGIS Online or an instance of ArcGIS Enterprise with Business Analyst.

### The `GIS` Source

Geoenrichment can access the necessary resources using either local or remote resources. Local resources require an installation of ArcGIS Pro with the Business Analyst extension and at least one data pack available. Remote resources require a Web GIS, either ArcGIS Online or ArcGIS Enterprise with Business Analyst.

If using a Web GIS, you must be logged in. The Geoenrichment module does _not_ support anonymous users. If the Web GIS is ArcGIS Online, the logged in user must have permissions to use credits since geoenrichment and report generation consume credits.

### Data Frames

The Geoenrichment module uses Pandas data frames as the standard data object for both inputs and outputs where ever it is practical.

## Getting Started

First, a few resources need to be loaded from the ArcGIS package.

In [1]:
import os

from arcgis.gis import GIS
from arcgis import geoenrichment
import pandas as pd

### Source

The Geoenrichment module uses a `GIS` source. This source can be either local (ArcGIS Pro + Business Analyst + data pack) or a Web GIS (ArcGIS Online or ArcGIS Enterprise + Business Analyst).

#### Local Source

If using a local source, this is specified using a `GIS` object created using the `'pro'` keyword.

In [2]:
gis_local = GIS('pro')

#### Web GIS Source

If using a remote Web GIS source, we must log in using any of the other available methods. In this case, we are using a username and password.

In [3]:
gis_agol = GIS(os.getenv('AGOL_URL'), username=os.getenv('AGOL_USERNAME'), password=os.getenv('AGOL_PASSWORD'))

## Discover Countries

Depending on what source you are using, the number of countries you have accessible will vary. Since most data is organized and varies by counry, the first step frequently is discovering what countries are available. The `get_countries` method provides a way to introspectively discover what countries you have available to work with. As is evident below, quite a few more countries are available using ArcGIS Online.

In [4]:
geoenrichment.get_countries(gis_local)

,iso2,iso3,country_name,vintage,country_id,data_source_id
0,CA,CAN,Canada,2021,CAN_ESRI_2021,LOCAL;;CAN_ESRI_2021
1,JP,JPN,Japan,2020,JAPAN2020,LOCAL;;JAPAN2020
2,US,USA,United States,2019,USA_ESRI_2019,LOCAL;;USA_ESRI_2019
4,US,USA,United States,2020,USA_PL_2020,LOCAL;;USA_PL_2020
3,US,USA,United States,2021,USA_ESRI_2021,LOCAL;;USA_ESRI_2021


In [5]:
geoenrichment.get_countries(gis_agol)

,iso2,iso3,country_name,datasets,default_dataset,alt_name,continent
0,AL,ALB,Albania,[ALB_MBR_2020],ALB_MBR_2020,ALBANIA,Europe
1,DZ,DZA,Algeria,[DZA_MBR_2021],DZA_MBR_2021,ALGERIA,Africa
2,AD,AND,Andorra,[AND_MBR_2020],AND_MBR_2020,ANDORRA,Europe
3,AO,AGO,Angola,[AGO_MBR_2021],AGO_MBR_2021,ANGOLA,Africa
4,AI,AIA,Anguilla,[AIA_MBR_2020],AIA_MBR_2020,ANGUILLA,North America
...,...,...,...,...,...,...,...
149,UZ,UZB,Uzbekistan,[UZB_MBR_2020],UZB_MBR_2020,UZBEKISTAN,Asia
150,VE,VEN,Venezuela,[VEN_MBR_2020],VEN_MBR_2020,"VENEZUELA, BOLIVARIAN REPUBLIC OF",South America
151,VN,VNM,Vietnam,[VNM_MBR_2020],VNM_MBR_2020,VIET NAM,Asia
152,VI,VIR,Virgin Islands,[VIR_MBR_2020],VIR_MBR_2020,UNITED STATES VIRGIN ISLANDS,North America


## Instantiate a Country Object

As discussed above, most data is organized by country, so a `Country` object needs to be created. This is how we are going to access most of the other capabilities.

In [6]:
usa_local = geoenrichment.Country('usa', gis=gis_local)

usa_local

<Country - United States 2021 ('local')>

In [7]:
usa_agol = geoenrichment.Country('usa', gis=gis_agol)

usa_agol

<Country - United States (GIS @ https://bateam.maps.arcgis.com version:10.1)>

## Enrich

Geoenrichment, accessed through the `Country.enrich` method, is how you access the demographic variables Esri makes available. The process of getting demographics is based on a geometric area typically referred to as a *polygon geometry*. 

This polygon geometry can be defined in a few ways. First, polygon geometries can be provided. Second, standard geographies such as block groups or zip codes can be explicitly defined using their unique identifier (ID). Third, points or line geometries can be provided, and the method of defining an area surrounding the point or line can be specified.

Before enriching though, we first need to select variables from the thousands of variables available. This is accomplished by either using a pre-selected set, or using the introspection capabilities with Pandas data frame filtering to select variables.

### Enrich Variable Selection

In this case, an frequent place to start, a decent subset of variables, are key current year variables. After accessing the variables through a property of the `Country` object, this subset can quickly be identified using filtering to identify patterns in the naming conventions.

Since one of the key concepts in the design of the Geoenrichment module is use of Pandas data frames as the building block, the data frame returned from introspection can be passed directly into the `Country.enrich` method later in the workflow.

In [8]:
ev = usa_agol.enrich_variables

ev.info()
ev.sample(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19153 entries, 0 to 19152
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   name               19153 non-null  object
 1   alias              19153 non-null  object
 2   data_collection    19153 non-null  object
 3   enrich_name        19153 non-null  object
 4   enrich_field_name  19153 non-null  object
 5   description        19060 non-null  object
 6   vintage            19051 non-null  object
 7   units              19153 non-null  object
dtypes: object(8)
memory usage: 1.2+ MB


,name,alias,data_collection,enrich_name,enrich_field_name,description,vintage,units
17330,MP29062a_B,2021 Bought Chick-Fil-A/6 Mo,restaurants,restaurants.MP29062a_B,restaurants_MP29062a_B,2021 Bought Chick-Fil-A Last 6 Mo,2021,count
16990,HISPASN_CY,2021 Hispanic Asian Pop,raceandhispanicorigin,raceandhispanicorigin.HISPASN_CY,raceandhispanicorigin_HISPASN_CY,2021 Hispanic Asian Population (Esri),2021,count
11340,X5013FY_A,2026 Avg: Men`s Sleepwear,clothing,clothing.X5013FY_A,clothing_X5013FY_A,2026 Men`s Sleepwear: Average,2026,currency
5286,MP16068h_I,2021 Index: HH Owns Electric Outdoor Grill,HouseholdGoodsFurnitureAppliances,HouseholdGoodsFurnitureAppliances.MP16068h_I,HouseholdGoodsFurnitureAppliances_MP16068h_I,2021 HH Owns Electric Outdoor Grill: Index,2021,count
16601,NHU18RBS20,2020 Non Hispanic Pop <18,nonhispanicbyagePL94,nonhispanicbyagePL94.NHU18RBS20,nonhispanicbyagePL94_NHU18RBS20,2020 Non Hispanic Population Age <18,2020,count


In [9]:
enrich_vars = ev[
    (ev.name.str.lower().str.endswith('cy'))
    & (ev.data_collection.str.lower().str.contains('key'))
].drop_duplicates('name').reset_index(drop=True)

enrich_vars.info()
enrich_vars

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   name               20 non-null     object
 1   alias              20 non-null     object
 2   data_collection    20 non-null     object
 3   enrich_name        20 non-null     object
 4   enrich_field_name  20 non-null     object
 5   description        20 non-null     object
 6   vintage            20 non-null     object
 7   units              20 non-null     object
dtypes: object(8)
memory usage: 1.4+ KB


,name,alias,data_collection,enrich_name,enrich_field_name,description,vintage,units
0,TOTPOP_CY,2021 Total Population,KeyUSFacts,KeyUSFacts.TOTPOP_CY,KeyUSFacts_TOTPOP_CY,2021 Total Population (Esri),2021,count
1,GQPOP_CY,2021 Group Quarters Population,KeyUSFacts,KeyUSFacts.GQPOP_CY,KeyUSFacts_GQPOP_CY,2021 Group Quarters Population (Esri),2021,count
2,DIVINDX_CY,2021 Diversity Index,KeyUSFacts,KeyUSFacts.DIVINDX_CY,KeyUSFacts_DIVINDX_CY,2021 Diversity Index (Esri),2021,count
3,TOTHH_CY,2021 Total Households,KeyUSFacts,KeyUSFacts.TOTHH_CY,KeyUSFacts_TOTHH_CY,2021 Total Households (Esri),2021,count
4,AVGHHSZ_CY,2021 Average Household Size,KeyUSFacts,KeyUSFacts.AVGHHSZ_CY,KeyUSFacts_AVGHHSZ_CY,2021 Average Household Size (Esri),2021,count
5,MEDHINC_CY,2021 Median Household Income,KeyUSFacts,KeyUSFacts.MEDHINC_CY,KeyUSFacts_MEDHINC_CY,2021 Median Household Income (Esri),2021,currency
6,AVGHINC_CY,2021 Average Household Income,KeyUSFacts,KeyUSFacts.AVGHINC_CY,KeyUSFacts_AVGHINC_CY,2021 Average Household Income (Esri),2021,currency
7,PCI_CY,2021 Per Capita Income,KeyUSFacts,KeyUSFacts.PCI_CY,KeyUSFacts_PCI_CY,2021 Per Capita Income (Esri),2021,currency
8,TOTHU_CY,2021 Total Housing Units,KeyUSFacts,KeyUSFacts.TOTHU_CY,KeyUSFacts_TOTHU_CY,2021 Total Housing Units (Esri),2021,count
9,OWNER_CY,2021 Owner Occupied HUs,KeyUSFacts,KeyUSFacts.OWNER_CY,KeyUSFacts_OWNER_CY,2021 Owner Occupied Housing Units (Esri),2021,count


### Enrich Polygons

Likely one of the most common input for `enrich` is pre-created polygons. Frequently these polygons are pre-generated study areas such as drive-time polygons around locations, and demographics are needed for creating and using forecasting models. As discussed earlier, we can directly use the enrich variable data frame as input to specify the enrichemnt varialbes we are interested in.

In [10]:
# load a few drive time polygons from arcgis online
dt_itm_id = 'de99cdc4c2f04ad79f448d2b653486f1' 
dt_df = gis_agol.content.get(dt_itm_id).layers[0].query(out_fields=['LOCNUM']).sdf

dt_df.info()
dt_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   OBJECTID  9 non-null      int64   
 1   LOCNUM    7 non-null      object  
 2   SHAPE     9 non-null      geometry
dtypes: geometry(1), int64(1), object(1)
memory usage: 344.0+ bytes


,OBJECTID,LOCNUM,SHAPE
0,1,229274261,"{""rings"": [[[-11666062.4440691, 4732166.253727..."
1,2,390328532,"{""rings"": [[[-11687341.4236634, 4821519.293239..."
2,3,403949338,"{""rings"": [[[-11673470.7705416, 4775047.686066..."
3,4,499359172,"{""rings"": [[[-11681041.3153383, 4837050.376413..."
4,5,570244483,"{""rings"": [[[-11687375.3871287, 4951434.845114..."


In [11]:
enrich_dt_df = usa_agol.enrich(dt_df, enrich_variables=enrich_vars)

enrich_dt_df.info()
enrich_dt_df

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 28 columns):
 #   Column                             Non-Null Count  Dtype   
---  ------                             --------------  -----   
 0   objectid                           9 non-null      int64   
 1   locnum                             7 non-null      object  
 2   source_country                     9 non-null      object  
 3   aggregation_method                 9 non-null      object  
 4   population_to_polygon_size_rating  9 non-null      float64 
 5   apportionment_confidence           9 non-null      float64 
 6   has_data                           9 non-null      int64   
 7   totpop_cy                          9 non-null      int64   
 8   gqpop_cy                           9 non-null      int64   
 9   divindx_cy                         9 non-null      float64 
 10  tothh_cy                           9 non-null      int64   
 11  avghhsz_cy                         9 non-null    

,objectid,locnum,source_country,aggregation_method,population_to_polygon_size_rating,apportionment_confidence,has_data,totpop_cy,gqpop_cy,divindx_cy,...,vacant_cy,medval_cy,avgval_cy,popgrw10_cy,hhgrw10_cy,famgrw10_cy,dpop_cy,dpopwrk_cy,dpopres_cy,SHAPE
0,1,229274261,USA,BlockApportionment:US.BlockGroups;PointsLayer:...,2.191,2.576,1,304286,3809,46.1,...,2665,422910,479649,1.92,1.86,1.81,317909,167443,150466,"{""rings"": [[[-104.7980219902777, 39.0732277478..."
1,2,390328532,USA,BlockApportionment:US.BlockGroups;PointsLayer:...,2.191,2.576,1,334981,1952,45.9,...,5314,472255,562111,0.90,0.93,0.82,326969,175809,151160,"{""rings"": [[[-104.98917431632918, 39.693619615..."
2,3,403949338,USA,BlockApportionment:US.BlockGroups;PointsLayer:...,2.191,2.576,1,446880,3694,44.6,...,7788,528572,623919,1.62,1.70,1.57,527086,320688,206398,"{""rings"": [[[-104.86457211862637, 39.371648317..."
3,4,499359172,USA,BlockApportionment:US.BlockGroups;PointsLayer:...,2.191,2.576,1,502750,9861,68.6,...,15337,537313,631549,1.72,1.75,1.53,605546,387223,218323,"{""rings"": [[[-104.93257947965587, 39.800891198..."
4,5,570244483,USA,BlockApportionment:US.BlockGroups;PointsLayer:...,2.191,2.576,1,178034,1352,40.3,...,1723,440561,491861,3.25,3.16,3.11,177980,88254,89726,"{""rings"": [[[-104.98947941474238, 40.585779193..."
5,6,634640569,USA,BlockApportionment:US.BlockGroups;PointsLayer:...,2.191,2.576,1,296756,1836,59.6,...,3455,444378,500175,2.05,2.13,2.06,272109,142113,129996,"{""rings"": [[[-104.86864877130206, 39.524001128..."
6,7,,USA,BlockApportionment:US.BlockGroups;PointsLayer:...,2.191,2.576,1,125791,555,37.6,...,2330,568686,671347,3.83,3.78,3.73,132421,69075,63346,"{""rings"": [[[-104.87648011776069, 39.582012539..."
7,8,None,USA,BlockApportionment:US.BlockGroups;PointsLayer:...,2.191,2.576,1,273842,3959,47.0,...,4263,487245,554477,1.55,1.52,1.48,270782,149260,121522,"{""rings"": [[[-104.97745795389609, 40.016594481..."
8,9,None,USA,BlockApportionment:US.BlockGroups;PointsLayer:...,2.191,2.576,1,348120,6847,60.4,...,4971,454140,508184,0.92,0.91,0.81,349988,191518,158470,"{""rings"": [[[-105.01544799921398, 39.731728019..."


### Enrich Standard Geographies

Standard geographies such as block groups and zip codes can be enriched by directly providing the IDs. Before enriching, we need to know how to tell the `Country.enrich` method what level of standard geography we are using. Fortuntely, there is an introspection method available for this, `Country.levels`. Using the `level_name` from the `levels` data frame, we can provide this into the enrich method with our ID list to perform geoenrichment.

In [12]:
# discover levels available, and how to correctly specify for intput into enrich
usa_local.levels

,level_name,alias,level_id,id_field,name_field,singular_name,plural_name,admin_level
0,block_groups,Block Groups,US.BlockGroups,ID,NAME,Block Group,Block Groups,Admin11
1,tracts,Census Tracts,US.Tracts,ID,NAME,Census Tract,Census Tracts,Admin10
2,places,Cities and Towns (Places),US.Places,ID,NAME,Place,Places,Admin9
3,zip5,ZIP Codes,US.ZIP5,ID,NAME,ZIP Code,ZIP Codes,Admin4
4,csd,County Subdivisions,US.CSD,ID,NAME,County Subdivision,County Subdivisions,Admin7
5,counties,Counties,US.Counties,ID,NAME,County,Counties,Admin3
6,cbsa,CBSAs,US.CBSA,ID,NAME,CBSA,CBSAs,Admin5
7,cd,Congressional Districts,US.CD,ID,NAME,Congressional District,Congressional Districts,Admin8
8,dma,DMAs,US.DMA,ID,NAME,DMA,DMAs,Admin6
9,states,States,US.States,ID,NAME,State,States,Admin2


In [13]:
# list of zip codes to enrich
std_geo = ['98501', '98502', '98506', '98512', '98513']

In [14]:
enrich_std_df = usa_agol.enrich(std_geo, enrich_variables=enrich_vars, standard_geography_level='zip5', standard_geography_id_column='zip_id')

enrich_std_df.info()
enrich_std_df

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 29 columns):
 #   Column                             Non-Null Count  Dtype   
---  ------                             --------------  -----   
 0   std_geography_level                5 non-null      object  
 1   std_geography_name                 5 non-null      object  
 2   std_geography_id                   5 non-null      object  
 3   source_country                     5 non-null      object  
 4   aggregation_method                 5 non-null      object  
 5   population_to_polygon_size_rating  5 non-null      float64 
 6   apportionment_confidence           5 non-null      float64 
 7   has_data                           5 non-null      int64   
 8   totpop_cy                          5 non-null      int64   
 9   gqpop_cy                           5 non-null      int64   
 10  divindx_cy                         5 non-null      float64 
 11  tothh_cy                           5 non-null    

,std_geography_level,std_geography_name,std_geography_id,source_country,aggregation_method,population_to_polygon_size_rating,apportionment_confidence,has_data,totpop_cy,gqpop_cy,...,vacant_cy,medval_cy,avgval_cy,popgrw10_cy,hhgrw10_cy,famgrw10_cy,dpop_cy,dpopwrk_cy,dpopres_cy,SHAPE
0,US.ZIP5,Olympia,98501,USA,Query:US.ZIP5,2.191,2.576,1,45608,274,...,894,394653,439413,1.45,1.42,1.38,60047,37487,22560,"{""rings"": [[[-122.90437000028668, 47.075349999..."
1,US.ZIP5,Olympia,98502,USA,Query:US.ZIP5,2.191,2.576,1,36141,906,...,1153,437010,535312,1.51,1.66,1.43,37001,18659,18342,"{""rings"": [[[-122.94492773701208, 47.187828655..."
2,US.ZIP5,Olympia,98506,USA,Query:US.ZIP5,2.191,2.576,1,19620,386,...,377,404730,477796,0.79,0.79,0.62,18992,9207,9785,"{""rings"": [[[-122.84836000010098, 47.162459999..."
3,US.ZIP5,Olympia,98512,USA,Query:US.ZIP5,2.191,2.576,1,32100,797,...,596,388747,418532,1.22,1.11,0.93,28636,13426,15210,"{""rings"": [[[-123.15327999980738, 47.054389999..."
4,US.ZIP5,Olympia,98513,USA,Query:US.ZIP5,2.191,2.576,1,36053,86,...,665,335359,417275,1.37,1.24,1.14,27483,8875,18608,"{""rings"": [[[-122.70325000018195, 47.069840000..."


### Enrich Point or Line Geometries

Point and line geometries can also be used as inputs, but the method of deteriming the area to enrich around the geometries must be determined. If nothing else is provided, the default of one kilometer in a straight line will be used. Also, if using line geometry, a straight line distance (buffer) is the only option.

In this case, we are going to demonstrate how to use a few points to enrich using a five-minute drive time surrounding these locations. Similar to above, we need to know the correct way to let `Country.enrich` know the right method to apply, and this is discovered through `Country.travel_modes`. The values from the `name` column is used.

In [15]:
usa_agol.travel_modes

,name,alias,description,type,impedance,impedance_category,time_attribute_name,distance_attribute_name,travel_mode_id,travel_mode_dict
0,driving_time,Driving Time,Models the movement of cars and other similar ...,AUTOMOBILE,TravelTime,temporal,TravelTime,Kilometers,FEgifRtFndKNcJMJ,"{""attributeParameterValues"": [{""attributeName""..."
1,driving_distance,Driving Distance,Models the movement of cars and other similar ...,AUTOMOBILE,Kilometers,distance,TravelTime,Kilometers,iKjmHuBSIqdEfOVr,"{""attributeParameterValues"": [{""attributeName""..."
2,trucking_time,Trucking Time,Models basic truck travel by preferring design...,TRUCK,TruckTravelTime,temporal,TruckTravelTime,Kilometers,ZzzRtYcPLjXFBKwr,"{""attributeParameterValues"": [{""attributeName""..."
3,trucking_distance,Trucking Distance,Models basic truck travel by preferring design...,TRUCK,Kilometers,distance,TruckTravelTime,Kilometers,UBaNfFWeKcrRVYIo,"{""attributeParameterValues"": [{""attributeName""..."
4,walking_time,Walking Time,Follows paths and roads that allow pedestrian ...,WALK,WalkTime,temporal,WalkTime,Kilometers,caFAgoThrvUpkFBW,"{""attributeParameterValues"": [{""attributeName""..."
5,walking_distance,Walking Distance,Follows paths and roads that allow pedestrian ...,WALK,Kilometers,distance,WalkTime,Kilometers,yFuMFwIYblqKEefX,"{""attributeParameterValues"": [{""attributeName""..."
6,rural_driving_time,Rural Driving Time,Models the movement of cars and other similar ...,AUTOMOBILE,TravelTime,temporal,TravelTime,Kilometers,NmNhNDUwZmE1YTlj,"{""attributeParameterValues"": [{""attributeName""..."
7,rural_driving_distance,Rural Driving Distance,Models the movement of cars and other similar ...,AUTOMOBILE,Kilometers,distance,TravelTime,Kilometers,Yzk3NjI1NTU5NjVj,"{""attributeParameterValues"": [{""attributeName""..."


In [16]:
pt_itm_id = 'b062497f7b634295b28c2fd9dc398e4b'
pt_df = gis_agol.content.get(pt_itm_id).layers[0].query(out_fields='GlobalID').sdf.iloc[:5]
pt_df.spatial.set_geometry('SHAPE')

pt_df.info()
pt_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   ObjectId  5 non-null      int64   
 1   GlobalID  5 non-null      object  
 2   SHAPE     5 non-null      geometry
dtypes: geometry(1), int64(1), object(1)
memory usage: 248.0+ bytes


,ObjectId,GlobalID,SHAPE
0,1,9c2b7b1e-0767-4877-85f7-38e1cdb766ed,"{""x"": -10631577.805514993, ""y"": 3484563.070506..."
1,2,b6a627de-94ca-4f4d-b340-6f735e766800,"{""x"": -10026505.893004062, ""y"": 3497836.290345..."
2,3,9780d0a4-7c01-4f3b-b235-e11b436a72ef,"{""x"": -17591320.05138795, ""y"": 2451176.7408811..."
3,4,e906e3f1-b78b-4fe1-ba98-a86d39374221,"{""x"": -8990880.643777844, ""y"": 4210546.6868309..."
4,5,7d8d000c-f33d-4d2e-9537-09760643716f,"{""x"": -8907264.444289908, ""y"": 4946790.1852645..."


In [17]:
enrich_pt_df = usa_agol.enrich(pt_df, enrich_variables=enrich_vars, proximity_type='driving_time', proximity_value=5, proximity_metric='minutes')

enrich_pt_df.info()
enrich_pt_df

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 32 columns):
 #   Column                             Non-Null Count  Dtype   
---  ------                             --------------  -----   
 0   object_id                          5 non-null      int64   
 1   global_id                          5 non-null      object  
 2   source_country                     5 non-null      object  
 3   area_type                          5 non-null      object  
 4   buffer_units                       5 non-null      object  
 5   buffer_units_alias                 5 non-null      object  
 6   buffer_radii                       5 non-null      int64   
 7   aggregation_method                 5 non-null      object  
 8   population_to_polygon_size_rating  5 non-null      float64 
 9   apportionment_confidence           5 non-null      float64 
 10  has_data                           5 non-null      int64   
 11  totpop_cy                          5 non-null    

,object_id,global_id,source_country,area_type,buffer_units,buffer_units_alias,buffer_radii,aggregation_method,population_to_polygon_size_rating,apportionment_confidence,...,vacant_cy,medval_cy,avgval_cy,popgrw10_cy,hhgrw10_cy,famgrw10_cy,dpop_cy,dpopwrk_cy,dpopres_cy,SHAPE
0,1,9c2b7b1e-0767-4877-85f7-38e1cdb766ed,USA,NetworkServiceArea,Minutes,Drive Time Minutes,5,BlockApportionment:US.BlockGroups;PointsLayer:...,2.191,2.576,...,2016,212612,252693,0.64,0.50,0.33,44292,29418,14874,"{""x"": -95.50508837, ""y"": 29.852179949999996, ""..."
1,2,b6a627de-94ca-4f4d-b340-6f735e766800,USA,NetworkServiceArea,Minutes,Drive Time Minutes,5,BlockApportionment:US.BlockGroups;PointsLayer:...,2.191,2.576,...,3193,421893,610191,3.17,3.71,3.07,59047,52520,6527,"{""x"": -90.0696349, ""y"": 29.95554066999999, ""sp..."
2,3,9780d0a4-7c01-4f3b-b235-e11b436a72ef,USA,NetworkServiceArea,Minutes,Drive Time Minutes,5,BlockApportionment:US.BlockGroups;PointsLayer:...,2.191,2.576,...,606,670743,702760,-0.18,-0.41,-0.42,12920,5554,7366,"{""x"": -158.0255167, ""y"": 21.496469949999998, ""..."
3,4,e906e3f1-b78b-4fe1-ba98-a86d39374221,USA,NetworkServiceArea,Minutes,Drive Time Minutes,5,BlockApportionment:US.BlockGroups;PointsLayer:...,2.191,2.576,...,559,260935,294294,1.34,1.68,1.00,19786,11157,8629,"{""x"": -80.766455, ""y"": 35.34267, ""spatialRefer..."
4,5,7d8d000c-f33d-4d2e-9537-09760643716f,USA,NetworkServiceArea,Minutes,Drive Time Minutes,5,BlockApportionment:US.BlockGroups;PointsLayer:...,2.191,2.576,...,137,238951,261542,-0.03,0.11,-0.06,7928,4683,3245,"{""x"": -80.0153179, ""y"": 40.55408534999999, ""sp..."


## Viewing Available Reports

To look up what reports are available for a given country, the `Country.reports` property returns a data frame of report ids along with metadata and exportable formats.

In [18]:
reports = usa_agol.reports

reports.info()
reports.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53 entries, 0 to 52
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   id          53 non-null     object
 1   title       53 non-null     object
 2   categories  53 non-null     object
 3   formats     53 non-null     object
dtypes: object(4)
memory usage: 1.8+ KB


,id,title,categories,formats
0,census2010_profile,2010 Census Profile,[Demographics],"[pdf, xlsx]"
1,acs_housing,ACS Housing Summary,[Demographics],"[pdf, xlsx]"
2,acs_keyfacts,ACS Key Population & Household Facts,[Demographics],"[pdf, xlsx]"
3,acs_population,ACS Population Summary,[Demographics],"[pdf, xlsx]"
4,55plus,Age 50+ Profile,[Demographics],"[pdf, xlsx]"


### Creating a Report


The Create Report method allows you to create many types of high quality reports for a variety of use cases describing the input area. If a point is used as a study area, the service will create a 1-mile ring buffer around the point to collect and append enrichment data. Optionally, you can create a buffer ring or drive-time service area around points of interest to generate PDF or Excel reports containing relevant information for the area on demographics, consumer spending, tapestry market, business or market potential.

Report options are available and can be used to describe and gain a better understanding about the market, customers/clients and competition associated with an area of interest.

In [19]:
study_area = [{"geometry":{"rings":[[[-117.26,32.81],[-117.40,32.92],[-117.12,32.80],[-117.26,32.81]]],
                      "spatialReference":{"wkid":4326}},"attributes":{"id":"Polygon 1","name":"Optional Name 1"}
             },
            {"address":{"text":"380 New York St. Redlands, CA 92373"}}]

r = geoenrichment.create_report(study_areas=study_area,
                     report="census2010_profile",
                     export_format="PDF",
                     use_data={"sourceCountry":"US"},
                     out_folder=r"c:\temp", out_name="report.pdf")
print(r)

c:\temp\report.pdf


## Standard geography query

The GeoEnrichment service provides a helper method that returns standard geography IDs and features for the supported geographic levels in the United States and Canada.

As indicated throughout this documentation guide, the GeoEnrichment service uses the concept of a study area to define the location of the point or area that you want to enrich with additional information. Locations can also be passed as one or many named statistical areas. This form of a study area lets you define an area by the ID of a standard geographic statistical feature, such as a census or postal area. For example, to obtain enrichment information for a U.S. state, county or ZIP Code or a Canadian province or postal code, the Standard Geography Query helper method allows you to search and query standard geography areas so that they can be used in the GeoEnrichment method to obtain facts about the location.

The most common workflow for this service is to find a FIPS (standard geography ID) for a geographic name. For example, you can use this service to find the FIPS for the county of San Diego which is 06073. You can then use this FIPS ID within the GeoEnrichment service study area definition to get geometry and optional demographic data for the county. This study area definition is passed as a parameter to the GeoEnrichment service to return data defined in the enrichment pack and optionally return geometry for the feature.


In [20]:
df = geoenrichment.standard_geography_query(source_country='US',
                            layers=['US.States'],
                            ids=['06'],
                            return_geometry=True)

df

,DatasetID,Hierarchy,DataLayerID,AreaID,AreaName,MajorSubdivisionName,MajorSubdivisionAbbr,MajorSubdivisionType,CountryAbbr,ObjectId,Score,SHAPE
0,USA_ESRI_2021,census,US.States,06,California,California,CA,State,US,1,100,"{""rings"": [[[-122.30932600010335, 42.008461000..."


## Summary

The geoenrichment module provides access to the thousands of demographic variables available with the introspection methods needed to provide the correct inputs for the parameters. Also, the geoenrichment module provides access to automated reporting and standard geographies.